# NUTS-2 Regional Waste Hotspot Analysis

This notebook allocates national waste generation to NUTS-2 regions using SBS employment data as a proxy, then applies clustering to identify regional hotspots for waste recovery.

## Methodology
1. **Data sources**: 
   - Waste generation by country × NACE × waste type (Eurostat env_wasgen)
   - Employment by NUTS-2 region × NACE (Eurostat SBS sbs_r_nuts06_r2)
   - Recycling potential index by waste category

2. **Allocation approach**:
   - For each country-NACE combination, calculate regional employment shares
   - Allocate national waste to regions proportionally to employment
   - This assumes waste generation correlates with economic activity

3. **Analysis**:
   - Cluster regions by waste profile (not aggregated totals)
   - Calculate economic potential using recycling indices
   - Identify hotspots by region, NACE sector, and waste type

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import eurostat
import warnings
warnings.filterwarnings('ignore')

pd.options.display.max_columns = 100
pd.options.display.max_rows = 100
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Data

In [ ]:
# Load waste generation data (country × NACE × waste type)
wasgen = pd.read_csv('../data/interim/Generated_waste_per_nace_country.csv')
print(f"Waste generation records: {len(wasgen):,}")
wasgen.head()

In [ ]:
# Load SBS NUTS-2 employment data
# If not cached, fetch from Eurostat
try:
    sbs = pd.read_csv('../data/interim/sbs_nuts2_raw.csv')
    print("Loaded SBS data from cache")
except FileNotFoundError:
    print("Fetching SBS data from Eurostat...")
    sbs = eurostat.get_data_df('sbs_r_nuts06_r2', flags=False)
    sbs.to_csv('../data/interim/sbs_nuts2_raw.csv', index=False)

# Fix geo column name if needed
geo_col = [c for c in sbs.columns if 'geo' in c.lower()][0]
if geo_col != 'geo':
    sbs = sbs.rename(columns={geo_col: 'geo'})

print(f"SBS records: {len(sbs):,}")
print(f"Columns: {list(sbs.columns)}")

# Fetch NUTS2 region names from the geo dimension
print("\nFetching NUTS2 region names...")
geo_labels_df = eurostat.get_dic('sbs_r_nuts06_r2', 'geo', frmt='df')

# Filter to NUTS2 codes (4 characters, starting with letters)
geo_labels_df = geo_labels_df[
    (geo_labels_df['val'].str.len() == 4) & 
    (geo_labels_df['val'].str[:2].str.isalpha())
]

# Create lookup dictionary
nuts2_name_map = dict(zip(geo_labels_df['val'], geo_labels_df['descr']))

# Add derived columns to sbs
sbs['nuts2_name'] = sbs['geo'].map(nuts2_name_map)
sbs['country_code'] = sbs['geo'].str[:2]
sbs['is_nuts2'] = sbs['geo'].str.len() == 4

# Get employment values (most recent year with data)
year_cols = [c for c in sbs.columns if str(c).isdigit()]
sbs['employment'] = sbs[year_cols].bfill(axis=1).iloc[:, 0]

print(f"NUTS2 names loaded: {len(nuts2_name_map)}")

# Show a few examples
sample_regions = ['DE11', 'FR10', 'SE11', 'ES30', 'ITC4']
print("\nSample region names:")
for code in sample_regions:
    if code in nuts2_name_map:
        print(f"  {code}: {nuts2_name_map[code]}")

In [ ]:
# Load recycling potential index
recyc_pot = pd.read_csv('../data/raw/EWC-recycling potential.csv', sep=';')
recyc_pot

## 2. Prepare Waste Generation Data

In [ ]:
# Country code mapping (ISO 2-letter)
country_map = {
    'Germany': 'DE', 'France': 'FR', 'Italy': 'IT', 'Spain': 'ES', 
    'Poland': 'PL', 'Netherlands': 'NL', 'Belgium': 'BE', 'Sweden': 'SE',
    'Austria': 'AT', 'Czechia': 'CZ', 'Portugal': 'PT', 'Greece': 'EL',
    'Hungary': 'HU', 'Denmark': 'DK', 'Finland': 'FI', 'Slovakia': 'SK',
    'Ireland': 'IE', 'Croatia': 'HR', 'Lithuania': 'LT', 'Slovenia': 'SI',
    'Latvia': 'LV', 'Estonia': 'EE', 'Cyprus': 'CY', 'Luxembourg': 'LU',
    'Malta': 'MT', 'Bulgaria': 'BG', 'Romania': 'RO', 'United Kingdom': 'UK',
    'Norway': 'NO', 'Iceland': 'IS', 'Türkiye': 'TR', 'Serbia': 'RS',
    'North Macedonia': 'MK', 'Montenegro': 'ME', 'Albania': 'AL',
    'Bosnia and Herzegovina': 'BA', 'Kosovo*': 'XK', 'Liechtenstein': 'LI'
}
wasgen['country_code'] = wasgen['country'].map(country_map)

In [ ]:
# Filter to sector-specific waste (exclude totals and aggregates)
exclude_wastes = ['TOTAL', 'PRIM', 'SEC', 'TOT_X_MIN', 'W12-13']
exclude_nace = ['TOTAL_HH', 'EP_HH', 'HH']  # Keep sector-specific only

wasgen_detail = wasgen[
    (~wasgen['waste'].isin(exclude_wastes)) &
    (~wasgen['waste'].str.contains('X_', na=False)) &
    (~wasgen['nace_r2'].isin(exclude_nace)) &
    (wasgen['mean_wasgen'] > 0) &
    (wasgen['country_code'].notna())
].copy()

print(f"Filtered records: {len(wasgen_detail):,}")
print(f"Countries: {wasgen_detail['country_code'].nunique()}")
print(f"NACE activities: {wasgen_detail['nace_r2'].nunique()}")
print(f"Waste types: {wasgen_detail['waste'].nunique()}")

In [ ]:
# Show available NACE codes in waste data
print("NACE codes in waste data:")
for nace in sorted(wasgen_detail['nace_r2'].unique()):
    activity = wasgen_detail[wasgen_detail['nace_r2'] == nace]['nace_r2_activity'].iloc[0]
    print(f"  {nace}: {activity[:60]}")

In [ ]:
# Show available waste types
print(f"\nWaste types ({wasgen_detail['waste'].nunique()}):")
waste_types = wasgen_detail[['waste', 'waste_description']].drop_duplicates().sort_values('waste')
for _, row in waste_types.iterrows():
    print(f"  {row['waste']}: {row['waste_description'][:55]}")

## 3. Build Allocation Keys from SBS Employment Data

In [ ]:
# SBS indicators available
indic_map = {
    'V11210': 'Turnover (€ million)',
    'V13320': 'Value added (€ million)', 
    'V16110': 'Number of persons employed',
    'V91290': 'Wages and salaries (€ million)',
    'V94310': 'Number of local units'
}
print("SBS indicators:")
for code, desc in indic_map.items():
    print(f"  {code}: {desc}")

In [ ]:
# Filter to employment indicator (V16110) and NUTS-2 regions with data
sbs_nuts2 = sbs[
    (sbs['indic_sb'] == 'V16110') & 
    (sbs['is_nuts2']) & 
    (sbs['employment'] > 0)
].copy()

print(f"NUTS-2 employment records: {len(sbs_nuts2):,}")
print(f"Unique NUTS-2 regions: {sbs_nuts2['geo'].nunique()}")
print(f"Countries covered: {sbs_nuts2['country_code'].nunique()}")

In [ ]:
# NACE code mapping: waste data aggregates -> SBS detailed codes
nace_expansion = {
    'C24_C25': ['C24', 'C25'],
    'C10-C12': ['C10', 'C11', 'C12'],
    'C13-C15': ['C13', 'C14', 'C15'],
    'C17_C18': ['C17', 'C18'],
    'C20-C22': ['C20', 'C21', 'C22'],
    'C26-C30': ['C26', 'C27', 'C28', 'C29', 'C30'],
    'C31-C33': ['C31', 'C32', 'C33'],
    'C31_C32': ['C31', 'C32'],
    'D': ['D35'],
    'E': ['E36', 'E37', 'E38', 'E39'],
    'E36_E37_E39': ['E36', 'E37', 'E39'],
    'G-U_X_G4677': ['G', 'H', 'I', 'J', 'K', 'L', 'M', 'N'],
}

print("NACE expansion mapping:")
for agg, detailed in nace_expansion.items():
    print(f"  {agg} -> {detailed}")

In [ ]:
def get_regional_shares(sbs_data, country_code, nace_codes):
    """Calculate employment shares for a country and list of NACE codes."""
    regional = sbs_data[
        (sbs_data['country_code'] == country_code) &
        (sbs_data['nace_r2'].isin(nace_codes))
    ].groupby('geo')['employment'].sum().reset_index()
    
    total = regional['employment'].sum()
    if total > 0:
        regional['share'] = regional['employment'] / total
    else:
        regional['share'] = 0
    return regional

# Test the function
test = get_regional_shares(sbs_nuts2, 'DE', ['C24', 'C25'])
print(f"Example: Germany metal manufacturing (C24+C25) - {len(test)} NUTS-2 regions")
test.head(10)

## 4. Allocate Waste to NUTS-2 Regions

In [ ]:
# Allocate each waste record to NUTS-2 regions based on employment share
results = []
missing_nace = set()

for _, row in wasgen_detail.iterrows():
    country_code = row['country_code']
    nace_code = row['nace_r2']
    waste_type = row['waste']
    waste_desc = row['waste_description']
    nace_activity = row['nace_r2_activity']
    total_waste = row['mean_wasgen']
    
    # Expand NACE codes if aggregated
    sbs_nace_list = nace_expansion.get(nace_code, [nace_code])
    
    # Get regional employment shares
    regional_shares = get_regional_shares(sbs_nuts2, country_code, sbs_nace_list)
    
    if len(regional_shares) == 0 or regional_shares['share'].sum() == 0:
        missing_nace.add((country_code, nace_code))
        continue
    
    # Allocate waste to regions
    for _, reg in regional_shares.iterrows():
        allocated = total_waste * reg['share']
        if allocated > 0:
            # Look up NUTS2 region name
            nuts2_name = nuts2_name_map.get(reg['geo'], reg['geo'])
            results.append({
                'nuts2_region': reg['geo'],
                'nuts2_name': nuts2_name,
                'country_code': country_code,
                'nace_r2': nace_code,
                'nace_activity': nace_activity,
                'waste': waste_type,
                'waste_description': waste_desc,
                'allocated_waste_tonnes': allocated,
                'employment': reg['employment'],
                'allocation_share': reg['share']
            })

regional_waste = pd.DataFrame(results)
print(f"Allocated records: {len(regional_waste):,}")
print(f"Unique NUTS-2 regions: {regional_waste['nuts2_region'].nunique()}")
print(f"Missing NACE mappings: {len(missing_nace)}")

In [ ]:
# View the allocated data
regional_waste.head(20)

In [ ]:
# Summary statistics
print(f"Total allocated waste: {regional_waste['allocated_waste_tonnes'].sum()/1e9:.2f} billion tonnes")
print(f"\nBy country:")
country_totals = regional_waste.groupby('country_code')['allocated_waste_tonnes'].sum().sort_values(ascending=False)
for country, total in country_totals.head(10).items():
    print(f"  {country}: {total/1e9:.2f}B tonnes")

## 5. Explore the Detailed Data (by Region × NACE × Waste)

In [ ]:
# Top combinations by allocated waste (with descriptions)
print("Top 30 Region × NACE × Waste combinations by tonnage:")
top_combos = regional_waste.nlargest(30, 'allocated_waste_tonnes')[[
    'nuts2_region', 'nuts2_name', 'country_code', 'nace_r2', 'nace_activity', 
    'waste', 'waste_description', 'allocated_waste_tonnes'
]].copy()
top_combos['waste_Mt'] = top_combos['allocated_waste_tonnes'] / 1e6
top_combos[['nuts2_region', 'nuts2_name', 'nace_r2', 'waste', 'waste_Mt']]

In [ ]:
# Explore a specific region in detail
region_code = 'FR10'  # Île-de-France
region_data = regional_waste[regional_waste['nuts2_region'] == region_code]

print(f"\n{region_code} - Waste by NACE sector:")
by_nace = region_data.groupby(['nace_r2', 'nace_activity'])['allocated_waste_tonnes'].sum().sort_values(ascending=False)
for (nace, activity), tonnes in by_nace.head(10).items():
    print(f"  {nace}: {activity[:40]} - {tonnes/1e6:.1f}M tonnes")

print(f"\n{region_code} - Waste by type:")
by_waste = region_data.groupby(['waste', 'waste_description'])['allocated_waste_tonnes'].sum().sort_values(ascending=False)
for (waste, desc), tonnes in by_waste.head(10).items():
    print(f"  {waste}: {desc[:40]} - {tonnes/1e6:.1f}M tonnes")

In [ ]:
# Explore a specific NACE sector across regions
nace_code = 'C24_C25'  # Metal manufacturing
nace_data = regional_waste[regional_waste['nace_r2'] == nace_code]

print(f"\nTop regions for {nace_code} (Basic metals & Fabricated metal products):")
by_region = nace_data.groupby('nuts2_region')['allocated_waste_tonnes'].sum().sort_values(ascending=False)
for region, tonnes in by_region.head(15).items():
    country = region[:2]
    print(f"  {region} ({country}): {tonnes/1e6:.2f}M tonnes")

In [ ]:
# Explore a specific waste type across regions
waste_code = 'W061'  # Ferrous metal wastes
waste_data = regional_waste[regional_waste['waste'] == waste_code]

print(f"\nTop regions for {waste_code} (Metallic wastes, ferrous):")
by_region = waste_data.groupby('nuts2_region')['allocated_waste_tonnes'].sum().sort_values(ascending=False)
for region, tonnes in by_region.head(15).items():
    country = region[:2]
    print(f"  {region} ({country}): {tonnes/1e6:.2f}M tonnes")

## 6. Calculate Economic Potential

In [ ]:
# Recycling potential index (€/tonne)
waste_value_map = {
    'W061': 1000,   # Ferrous metals
    'W062': 1000,   # Non-ferrous metals
    'W063': 500,    # Mixed metals
    'W06': 800,     # All metallic wastes
    'W071': 1000,   # Glass
    'W072': 1000,   # Paper/cardboard
    'W073': 100,    # Rubber
    'W074': 100,    # Plastics
    'W075': 100,    # Wood
    'W076': 10,     # Textiles
    'W077': 1,      # PCB wastes
    'W08A': 300,    # WEEE
    'W081': 500,    # Discarded vehicles
    'W091': 10,     # Animal food waste
    'W092': 100,    # Green waste
    'W093': 10,     # Slurry/manure
    'W101': 10,     # Mixed municipal
    'W102': 10,     # Mixed/undifferentiated
    'W103': 10,     # Sorting residues
    'W10': 10,      # Mixed wastes total
    'W11': 10,      # Sludges
    'W121': 100,    # Construction mineral
    'W124': 10,     # Combustion wastes
    'W126': 100,    # Soils
    'W127': 10,     # Dredging spoils
    'W12A': 50,     # Mineral wastes
    'W12B': 50,     # Other mineral
    'W128_13': 50,  # Mineral treatment
    'W13': 50,      # Solidified wastes
    'W01-05': 50,   # Chemical/medical
    'W011': 100,    # Spent solvents
    'W012': 10,     # Acid/alkaline
    'W013': 100,    # Used oils
    'W02A': 10,     # Chemical wastes
    'W032': 10,     # Industrial sludges
    'W033': 10,     # Sludges from treatment
    'W05': 1,       # Health care
    'W06_07A': 500, # Recyclables
}

# Add economic potential to regional waste data
regional_waste['recycling_potential_eur_t'] = regional_waste['waste'].map(waste_value_map).fillna(10)
regional_waste['economic_potential_eur'] = regional_waste['allocated_waste_tonnes'] * regional_waste['recycling_potential_eur_t']

print(f"Total economic potential: €{regional_waste['economic_potential_eur'].sum()/1e9:.1f} billion/year")

In [ ]:
# Top combinations by economic potential (with descriptions)
print("Top 30 Region × NACE × Waste combinations by economic potential:")
top_econ = regional_waste.nlargest(30, 'economic_potential_eur')[[
    'nuts2_region', 'nuts2_name', 'country_code', 'nace_r2', 'nace_activity',
    'waste', 'waste_description', 'allocated_waste_tonnes', 
    'recycling_potential_eur_t', 'economic_potential_eur'
]].copy()
top_econ['waste_Mt'] = top_econ['allocated_waste_tonnes'] / 1e6
top_econ['econ_pot_M'] = top_econ['economic_potential_eur'] / 1e6
top_econ[['nuts2_region', 'nuts2_name', 'nace_r2', 'waste', 'waste_Mt', 'recycling_potential_eur_t', 'econ_pot_M']]

## 7. Create Pivot Tables for Analysis

In [ ]:
# Region × Waste type matrix (with descriptions)
# Create lookup dictionaries for descriptions
waste_desc_map = regional_waste[['waste', 'waste_description']].drop_duplicates().set_index('waste')['waste_description'].to_dict()
nace_desc_map = regional_waste[['nace_r2', 'nace_activity']].drop_duplicates().set_index('nace_r2')['nace_activity'].to_dict()
nuts2_desc_map = regional_waste[['nuts2_region', 'nuts2_name']].drop_duplicates().set_index('nuts2_region')['nuts2_name'].to_dict()

# Region × Waste type matrix with MultiIndex columns (code + description)
region_waste_matrix = regional_waste.pivot_table(
    index='nuts2_region',
    columns='waste',
    values='allocated_waste_tonnes',
    aggfunc='sum'
).fillna(0)

# Add region names as additional column in index
region_waste_matrix['nuts2_name'] = region_waste_matrix.index.map(nuts2_desc_map)
region_waste_matrix = region_waste_matrix.reset_index().set_index(['nuts2_region', 'nuts2_name'])

# Rename columns to include descriptions
new_cols = {col: f"{col} - {waste_desc_map.get(col, '')[:40]}" for col in region_waste_matrix.columns if col in waste_desc_map}
region_waste_matrix = region_waste_matrix.rename(columns=new_cols)

print(f"Region × Waste matrix: {region_waste_matrix.shape}")
region_waste_matrix.head()

In [ ]:
# Region × NACE matrix (with descriptions)
region_nace_matrix = regional_waste.pivot_table(
    index='nuts2_region',
    columns='nace_r2',
    values='allocated_waste_tonnes',
    aggfunc='sum'
).fillna(0)

# Add region names as additional column in index
region_nace_matrix['nuts2_name'] = region_nace_matrix.index.map(nuts2_desc_map)
region_nace_matrix = region_nace_matrix.reset_index().set_index(['nuts2_region', 'nuts2_name'])

# Rename columns to include NACE descriptions
new_cols = {col: f"{col} - {nace_desc_map.get(col, '')[:40]}" for col in region_nace_matrix.columns if col in nace_desc_map}
region_nace_matrix = region_nace_matrix.rename(columns=new_cols)

print(f"Region × NACE matrix: {region_nace_matrix.shape}")
region_nace_matrix.head()

In [ ]:
# Region × Waste economic potential matrix (with descriptions)
region_econ_matrix = regional_waste.pivot_table(
    index='nuts2_region',
    columns='waste',
    values='economic_potential_eur',
    aggfunc='sum'
).fillna(0)

# Add region names as additional column in index
region_econ_matrix['nuts2_name'] = region_econ_matrix.index.map(nuts2_desc_map)
region_econ_matrix = region_econ_matrix.reset_index().set_index(['nuts2_region', 'nuts2_name'])

# Rename columns to include waste descriptions
new_cols = {col: f"{col} - {waste_desc_map.get(col, '')[:40]}" for col in region_econ_matrix.columns if col in waste_desc_map}
region_econ_matrix = region_econ_matrix.rename(columns=new_cols)

print(f"Region × Waste economic potential matrix: {region_econ_matrix.shape}")

## 8. Clustering by Region × NACE × Waste Profile

The goal is to identify hotspots at the granular level of **NUTS2 Region × NACE Activity × Waste Type**. This allows us to find specific industrial activities in specific regions that are hotspots for particular waste streams.

In [ ]:
# Create the clustering dataset at Region × NACE × Waste level
# Each row is a unique combination, features include waste tonnage and economic potential

# Start from the detailed allocation data
cluster_source = regional_waste.copy()

# Create a unique identifier for each Region × NACE × Waste combination
cluster_source['combo_id'] = (cluster_source['nuts2_region'] + '_' + 
                              cluster_source['nace_r2'] + '_' + 
                              cluster_source['waste'])

# Aggregate if there are any duplicates (shouldn't be, but just in case)
cluster_base = cluster_source.groupby(['nuts2_region', 'nuts2_name', 'country_code', 
                                        'nace_r2', 'nace_activity', 
                                        'waste', 'waste_description']).agg({
    'allocated_waste_tonnes': 'sum',
    'economic_potential_eur': 'sum',
    'recycling_potential_eur_t': 'first'
}).reset_index()

print(f"Total Region × NACE × Waste combinations: {len(cluster_base):,}")
print(f"Unique regions: {cluster_base['nuts2_region'].nunique()}")
print(f"Unique NACE activities: {cluster_base['nace_r2'].nunique()}")
print(f"Unique waste types: {cluster_base['waste'].nunique()}")

In [ ]:
# Create features for clustering
# We'll use: log-transformed waste tonnage, economic potential, and categorical encodings

# Log-transform to handle skewed distributions
cluster_base['log_waste'] = np.log10(cluster_base['allocated_waste_tonnes'] + 1)
cluster_base['log_econ'] = np.log10(cluster_base['economic_potential_eur'] + 1)

# One-hot encode NACE and waste categories for clustering
from sklearn.preprocessing import LabelEncoder

# Create label encoders for categorical variables
le_nace = LabelEncoder()
le_waste = LabelEncoder()
le_region = LabelEncoder()

cluster_base['nace_encoded'] = le_nace.fit_transform(cluster_base['nace_r2'])
cluster_base['waste_encoded'] = le_waste.fit_transform(cluster_base['waste'])
cluster_base['region_encoded'] = le_region.fit_transform(cluster_base['nuts2_region'])

# Features for clustering: continuous + categorical info
# Option 1: Cluster on continuous values only (waste amount, economic potential)
# Option 2: Include categorical encodings to group similar activities together

# We'll use continuous features for primary clustering
features_for_clustering = ['log_waste', 'log_econ', 'recycling_potential_eur_t']
X_cluster = cluster_base[features_for_clustering].values

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print(f"Clustering features: {features_for_clustering}")
print(f"Data shape: {X_scaled.shape}")

In [ ]:
# Find optimal number of clusters (using subsample for speed)
k_range = range(2, 12)
silhouettes = []
inertias = []

# Subsample for faster silhouette calculation (192k rows is too slow)
sample_size = 10000
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), size=min(sample_size, len(X_scaled)), replace=False)
X_sample = X_scaled[sample_idx]

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)  # Fit on full data
    inertias.append(km.inertia_)
    # Calculate silhouette on subsample only
    sample_labels = km.predict(X_sample)
    silhouettes.append(silhouette_score(X_sample, sample_labels))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, 'bo-')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(list(k_range), silhouettes, 'go-')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title(f'Silhouette Analysis (n={sample_size} subsample)')

best_k = list(k_range)[np.argmax(silhouettes)]
print(f"Best k by silhouette: {best_k} (score: {max(silhouettes):.3f})")
plt.tight_layout()
plt.show()

In [ ]:
# Apply clustering - using 5 clusters to identify hotspot tiers
# Cluster 0: Low volume, low value (background)
# Higher clusters: Increasingly significant hotspots

n_clusters = 5  # Adjust based on silhouette or domain knowledge

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_base['cluster'] = kmeans.fit_predict(X_scaled)

# Sort clusters by mean economic potential (so higher cluster = higher value)
cluster_means = cluster_base.groupby('cluster')['economic_potential_eur'].mean().sort_values()
cluster_rank_map = {old: new for new, old in enumerate(cluster_means.index)}
cluster_base['cluster'] = cluster_base['cluster'].map(cluster_rank_map)

print(f"Cluster distribution (sorted by economic potential):")
for c in range(n_clusters):
    subset = cluster_base[cluster_base['cluster'] == c]
    print(f"  Cluster {c}: {len(subset):,} combinations, "
          f"avg €{subset['economic_potential_eur'].mean()/1e6:.1f}M, "
          f"avg {subset['allocated_waste_tonnes'].mean()/1e3:.1f}k tonnes")

In [ ]:
# PCA for visualization of the Region × NACE × Waste combinations
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

cluster_base['pc1'] = X_pca[:, 0]
cluster_base['pc2'] = X_pca[:, 1]

print(f"PCA explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")

# Show PCA loadings - how each feature contributes to the principal components
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=features_for_clustering
)
print("\nPCA Loadings (feature contributions):")
print(loadings.round(3))

# Show top hotspots (highest cluster, sorted by economic potential)
top_cluster = n_clusters - 1
top_hotspots = cluster_base[cluster_base['cluster'] == top_cluster].nlargest(30, 'economic_potential_eur')

print(f"\nTop 30 Hotspots (Cluster {top_cluster} - Highest Value):")
print("-" * 100)
for i, (_, row) in enumerate(top_hotspots.iterrows(), 1):
    print(f"{i:2}. {row['nuts2_region']} | {row['nace_r2']:12} | {row['waste']:6} | "
          f"{row['allocated_waste_tonnes']/1e6:6.2f}M t | €{row['economic_potential_eur']/1e9:5.2f}B")
    print(f"    {row['nuts2_name'][:35]} | {row['nace_activity'][:30]} | {row['waste_description'][:30]}")

## 9. Visualize Hotspot Clusters (Region × NACE × Waste)

In [ ]:
# PCA scatter plot of Region × NACE × Waste combinations
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, n_clusters))  # Green=high value, Red=low

# Plot 1: All combinations colored by cluster
for c in range(n_clusters):
    mask = cluster_base['cluster'] == c
    axes[0].scatter(cluster_base[mask]['pc1'], cluster_base[mask]['pc2'],
                    c=[colors[c]], s=20, alpha=0.4, label=f'Cluster {c}')

axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[0].set_title('Region × NACE × Waste Combinations by Hotspot Cluster')
axes[0].legend(title='Hotspot Tier')
axes[0].grid(True, alpha=0.3)

# Plot 2: Focus on top clusters with labels
top_clusters = [n_clusters-1, n_clusters-2]  # Top 2 clusters
for c in top_clusters:
    mask = cluster_base['cluster'] == c
    subset = cluster_base[mask]
    axes[1].scatter(subset['pc1'], subset['pc2'],
                    c=[colors[c]], s=50, alpha=0.6, 
                    label=f'Cluster {c} ({len(subset)} combos)', edgecolors='black', linewidths=0.3)

# Annotate top 15 hotspots
top_15 = cluster_base.nlargest(15, 'economic_potential_eur')
for _, row in top_15.iterrows():
    label = f"{row['nuts2_region']}\n{row['nace_r2']}\n{row['waste']}"
    axes[1].annotate(label, (row['pc1'], row['pc2']), fontsize=6, alpha=0.8)

axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[1].set_title('Top Hotspot Clusters (with Region-NACE-Waste labels)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('nuts2_clustering_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Aggregate view: Top regions by number of high-value hotspots
top_cluster_threshold = n_clusters - 2  # Clusters 3 and 4 are "high value"

high_value_combos = cluster_base[cluster_base['cluster'] >= top_cluster_threshold]

# Count hotspots per region
region_hotspot_count = high_value_combos.groupby(['nuts2_region', 'nuts2_name', 'country_code']).agg({
    'nace_r2': 'nunique',  # Number of different NACE activities
    'waste': 'nunique',     # Number of different waste types
    'allocated_waste_tonnes': 'sum',
    'economic_potential_eur': 'sum',
    'cluster': 'count'      # Total number of hotspot combinations
}).rename(columns={
    'nace_r2': 'n_nace_activities',
    'waste': 'n_waste_types',
    'cluster': 'n_hotspot_combos'
}).reset_index().sort_values('economic_potential_eur', ascending=False)

print(f"Regions with high-value hotspots (clusters >= {top_cluster_threshold}):")
print(f"Total regions: {len(region_hotspot_count)}")
print(f"\nTop 20 regions by economic potential from hotspots:")
print("-" * 90)
for i, (_, r) in enumerate(region_hotspot_count.head(20).iterrows(), 1):
    print(f"{i:2}. {r['nuts2_region']} - {r['nuts2_name'][:30]} ({r['country_code']})")
    print(f"    {r['n_hotspot_combos']} hotspots across {r['n_nace_activities']} NACE × {r['n_waste_types']} waste types | €{r['economic_potential_eur']/1e9:.2f}B")

## 10. Cluster Profiles by Region × NACE × Waste

In [ ]:
# Detailed profile of each cluster
for c in range(n_clusters):
    subset = cluster_base[cluster_base['cluster'] == c]
    
    print(f"\n{'='*70}")
    print(f"CLUSTER {c}: {len(subset):,} Region × NACE × Waste combinations")
    print(f"{'='*70}")
    
    # Summary stats
    print(f"Total waste: {subset['allocated_waste_tonnes'].sum()/1e9:.2f}B tonnes")
    print(f"Total economic potential: €{subset['economic_potential_eur'].sum()/1e9:.1f}B")
    print(f"Avg waste per combo: {subset['allocated_waste_tonnes'].mean()/1e3:.1f}k tonnes")
    print(f"Avg economic value per combo: €{subset['economic_potential_eur'].mean()/1e6:.2f}M")
    
    # Top regions in this cluster
    top_regions = subset.groupby('nuts2_region')['economic_potential_eur'].sum().nlargest(5)
    print(f"\nTop 5 regions:")
    for region, value in top_regions.items():
        print(f"  {region}: €{value/1e9:.2f}B")
    
    # Top NACE activities in this cluster
    top_nace = subset.groupby('nace_r2')['economic_potential_eur'].sum().nlargest(5)
    print(f"\nTop 5 NACE activities:")
    for nace, value in top_nace.items():
        activity = subset[subset['nace_r2'] == nace]['nace_activity'].iloc[0]
        print(f"  {nace}: €{value/1e9:.2f}B - {activity[:40]}")
    
    # Top waste types in this cluster
    top_waste = subset.groupby('waste')['economic_potential_eur'].sum().nlargest(5)
    print(f"\nTop 5 waste types:")
    for waste, value in top_waste.items():
        desc = subset[subset['waste'] == waste]['waste_description'].iloc[0]
        print(f"  {waste}: €{value/1e9:.2f}B - {desc[:40]}")
    
    # Example hotspots from this cluster
    print(f"\nExample hotspots:")
    for _, row in subset.nlargest(3, 'economic_potential_eur').iterrows():
        print(f"  {row['nuts2_region']} | {row['nace_r2']} | {row['waste']} | €{row['economic_potential_eur']/1e6:.1f}M")

## 11. Drill-Down: Specific Waste Streams

In [ ]:
# Focus on high-value waste streams: metals
metal_wastes = ['W061', 'W062', 'W063', 'W06']
metals_data = regional_waste[regional_waste['waste'].isin(metal_wastes)]

metals_by_region = metals_data.groupby(['nuts2_region', 'nuts2_name', 'country_code']).agg({
    'allocated_waste_tonnes': 'sum',
    'economic_potential_eur': 'sum'
}).reset_index().sort_values('economic_potential_eur', ascending=False)

print("Top 20 regions for METAL WASTES:")
for i, (_, r) in enumerate(metals_by_region.head(20).iterrows(), 1):
    print(f"  {i:2}. {r['nuts2_region']} - {r['nuts2_name'][:30]} ({r['country_code']}): {r['allocated_waste_tonnes']/1e6:.2f}M t, €{r['economic_potential_eur']/1e9:.2f}B")

In [ ]:
# Focus on recyclables
recyclable_wastes = ['W071', 'W072', 'W074', 'W075']  # Glass, Paper, Plastics, Wood
recyclables_data = regional_waste[regional_waste['waste'].isin(recyclable_wastes)]

recyclables_by_region = recyclables_data.groupby(['nuts2_region', 'nuts2_name', 'country_code']).agg({
    'allocated_waste_tonnes': 'sum',
    'economic_potential_eur': 'sum'
}).reset_index().sort_values('economic_potential_eur', ascending=False)

print("Top 20 regions for RECYCLABLES (Glass, Paper, Plastics, Wood):")
for i, (_, r) in enumerate(recyclables_by_region.head(20).iterrows(), 1):
    print(f"  {i:2}. {r['nuts2_region']} - {r['nuts2_name'][:30]} ({r['country_code']}): {r['allocated_waste_tonnes']/1e6:.2f}M t, €{r['economic_potential_eur']/1e9:.2f}B")

In [ ]:
# Focus on construction waste
construction_wastes = ['W121', 'W126', 'W12A', 'W12B']
construction_data = regional_waste[regional_waste['waste'].isin(construction_wastes)]

construction_by_region = construction_data.groupby(['nuts2_region', 'nuts2_name', 'country_code']).agg({
    'allocated_waste_tonnes': 'sum',
    'economic_potential_eur': 'sum'
}).reset_index().sort_values('allocated_waste_tonnes', ascending=False)

print("Top 20 regions for CONSTRUCTION/MINERAL WASTES:")
for i, (_, r) in enumerate(construction_by_region.head(20).iterrows(), 1):
    print(f"  {i:2}. {r['nuts2_region']} - {r['nuts2_name'][:30]} ({r['country_code']}): {r['allocated_waste_tonnes']/1e6:.2f}M t, €{r['economic_potential_eur']/1e6:.0f}M")

## 12. Export Results

In [ ]:
# Save detailed allocation data with cluster assignments
cluster_base_export = cluster_base[[
    'nuts2_region', 'nuts2_name', 'country_code',
    'nace_r2', 'nace_activity',
    'waste', 'waste_description',
    'allocated_waste_tonnes', 'recycling_potential_eur_t', 'economic_potential_eur',
    'cluster'
]].copy()

cluster_base_export.to_csv('../data/processed/nuts2_waste_allocated_detail.csv', index=False)
print(f"Saved: data/processed/nuts2_waste_allocated_detail.csv ({len(cluster_base_export):,} records)")

# Save summary by region (aggregated from detailed data)
region_summary = cluster_base.groupby(['nuts2_region', 'nuts2_name', 'country_code']).agg({
    'allocated_waste_tonnes': 'sum',
    'economic_potential_eur': 'sum',
    'nace_r2': 'nunique',
    'waste': 'nunique',
    'cluster': lambda x: (x >= top_cluster_threshold).sum()  # Count of high-value hotspots
}).rename(columns={
    'nace_r2': 'n_nace_activities',
    'waste': 'n_waste_types',
    'cluster': 'n_high_value_hotspots'
}).reset_index()

# Add the dominant cluster for each region
dominant_cluster = cluster_base.groupby('nuts2_region')['cluster'].agg(
    lambda x: x.value_counts().index[0]
).rename('dominant_cluster')
region_summary = region_summary.merge(dominant_cluster, on='nuts2_region')

region_summary = region_summary.sort_values('economic_potential_eur', ascending=False)
region_summary.to_csv('../data/processed/nuts2_regional_hotspots.csv', index=False)
print(f"Saved: data/processed/nuts2_regional_hotspots.csv ({len(region_summary):,} regions)")

# Save pivot tables
region_waste_matrix.to_csv('../data/processed/nuts2_region_waste_matrix.csv')
region_nace_matrix.to_csv('../data/processed/nuts2_region_nace_matrix.csv')
print(f"Saved pivot matrices")

In [ ]:
# Summary statistics
print("\n" + "="*70)
print("ANALYSIS SUMMARY - Region × NACE × Waste Hotspot Clustering")
print("="*70)
print(f"Total Region × NACE × Waste combinations analyzed: {len(cluster_base):,}")
print(f"  - NUTS-2 regions: {cluster_base['nuts2_region'].nunique()}")
print(f"  - NACE activities: {cluster_base['nace_r2'].nunique()}")
print(f"  - Waste types: {cluster_base['waste'].nunique()}")
print(f"\nTotal waste allocated: {cluster_base['allocated_waste_tonnes'].sum()/1e9:.2f} billion tonnes")
print(f"Total economic potential: €{cluster_base['economic_potential_eur'].sum()/1e9:.1f} billion/year")

print(f"\nCluster distribution:")
for c in range(n_clusters):
    subset = cluster_base[cluster_base['cluster'] == c]
    print(f"  Cluster {c}: {len(subset):,} combos ({len(subset)/len(cluster_base)*100:.1f}%) - €{subset['economic_potential_eur'].sum()/1e9:.1f}B")

print(f"\nTop 10 Hotspots (Region × NACE × Waste):")
for i, (_, r) in enumerate(cluster_base.nlargest(10, 'economic_potential_eur').iterrows(), 1):
    print(f"  {i:2}. {r['nuts2_region']} | {r['nace_r2']} | {r['waste']} | €{r['economic_potential_eur']/1e9:.2f}B")
    print(f"      {r['nuts2_name'][:25]} | {r['nace_activity'][:25]} | {r['waste_description'][:25]}")